# 4.10 · Ноутбук 2. Baseline
Перш ніж будувати нейронну мережу, потрібна проста точка відліку. Deep learning виправданий, лише якщо він дає помітний приріст над простішою моделлю **за того самого протоколу**.

- **B-rule**: тривіальне правило «завжди найчастіший клас» (нижня межа);
- **B0**: логістична регресія на 64 пікселях (лінійна модель, стандартний baseline для зображень малої роздільності).

Оцінка тут іде тільки на validation. Test використовується лише в ноутбуці 3 для фінального порівняння.

In [ ]:
import os, json
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%config InlineBackend.figure_formats = ["jpeg"]
plt.rcParams["figure.dpi"] = 80
sns.set_theme(style="whitegrid")

CFG = json.load(open("../config.json", encoding="utf-8"))
SEED = CFG["seed"]

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
import time

digits = load_digits()
X, y = digits.images.reshape(len(digits.images), -1), digits.target
split = np.load("../data/split.npz")
Xtr, ytr, Xva, yva = X[split["train"]], y[split["train"]], X[split["val"]], y[split["val"]]
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)

In [ ]:
rows = []
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr_s, ytr)
rows.append({"run": "B-rule", "model": "most frequent class",
             "train_acc": accuracy_score(ytr, dummy.predict(Xtr_s)), "val_acc": accuracy_score(yva, dummy.predict(Xva_s))})

t0 = time.time()
b = CFG["baseline"]
logreg = LogisticRegression(C=b["C"], max_iter=b["max_iter"]).fit(Xtr_s, ytr)
rows.append({"run": "B0", "model": "LogisticRegression",
             "train_acc": accuracy_score(ytr, logreg.predict(Xtr_s)), "val_acc": accuracy_score(yva, logreg.predict(Xva_s)),
             "val_macro_f1": f1_score(yva, logreg.predict(Xva_s), average="macro"), "time_s": time.time() - t0})
base = pd.DataFrame(rows)
base["gap"] = base["train_acc"] - base["val_acc"]
os.makedirs("../results", exist_ok=True)
base.to_csv("../results/baseline_metrics.csv", index=False)
base.round(4)

**Висновок baseline.** Лінійна модель уже дає високу точність на validation. Це висока планка: CNN має перевершити її на тому самому протоколі, інакше складність DL не виправдана. Розрив train/val у логістичної регресії показує, наскільки вона підлаштовується під навчальні дані.